# 01 — Scrape Exeter Module Bank: Index Pages

**Purpose.** For every (department, year) pair, hit the department's module listing page on the Exeter Module Bank and collect every module code listed on it. The result is a master CSV of "which modules existed in which department in which year", which the next notebook uses to fetch individual module descriptors.

**Source.** `https://www.exeter.ac.uk/study/studyinformation/modules?prog=<slug>&year=<YYYY/Y>`

**Output.** `data/raw/module_bank/index_<year>.csv` (one file per year) and `data/raw/module_bank/index_all.csv` (concatenated).

## How this notebook handles Exeter's messy URL structure

A first-pass run showed two quirks:

1. **Slugs don't always match department names.** The Medical School is `med-school` (not `medical-sciences`); Sociology/Philosophy/Anthropology use three separate slugs `spa`/`sociology`/`anthropology`; some Business School subjects like Finance don't have their own slug at all and live inside `management`.
2. **Multiple slugs can return overlapping content.** `management` returns ~300 modules, which likely bundles BEE (Economics), BEM (Management), BEA (Accounting), BEF (Finance), BEP (People Management) all together. And `communications` returned 87 Drama-prefix modules — it's just Drama under another name.

This notebook handles both by:
- Probing a big candidate list against one recent year to find slugs that return data.
- De-duplicating on **the set of module-code prefixes** each slug returns (not just the modal one), so a slug only gets kept if it brings at least one prefix no other slug provides.
- Treating the three-letter **module-code prefix** as the real department identifier going forward. Analysis is at that level, not the slug level.


## 1. Imports and configuration

In [3]:
import os
import re
import time
from pathlib import Path

import requests
import pandas as pd
from bs4 import BeautifulSoup

# --- Paths ---
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RAW_DIR = PROJECT_ROOT / "data" / "raw" / "module_bank"
RAW_DIR.mkdir(parents=True, exist_ok=True)
print("Saving raw index files to:", RAW_DIR)

# --- Scraping config ---
BASE_URL = "https://www.exeter.ac.uk/study/studyinformation/modules"
HEADERS = {
    "User-Agent": "Mozilla/5.0 (Exeter BEE2041 student project; educational research)"
}
SLEEP_BETWEEN_REQUESTS = 1.0  # seconds — be polite to Exeter's servers

Saving raw index files to: C:\Users\kenna\datasci\files\data\raw\module_bank


## 2. Candidate list of department slugs

This list combines obvious guesses, slugs that worked on prior runs, and slugs verified by spot-checking Exeter's public pages. The discovery step below will whittle this down to a non-overlapping set.

In [5]:
CANDIDATE_SLUGS = [
    # --- Faculty of Environment, Science and Economy (ESE) ---
    "economics",
    "business", "business-school",
    "accounting", "finance", "management", "marketing",
    "mathematics", "maths",
    "computer-science", "computerscience", "cs",
    "engineering",
    "physics", "physics-astronomy",
    "geography",
    "biosciences", "biology",
    "naturalsciences", "natural-sciences",
    "renewable-energy", "environmental-sciences",

    # --- Faculty of Health and Life Sciences (HLS) ---
    "med-school", "medicine", "medical-sciences", "medicalsciences",
    "medical-imaging", "clinical-sciences",
    "sport-health-sciences", "sport-and-health-sciences",
    "sports-and-health-sciences", "sport-health",
    "psychology",
    "nursing", "nursing-midwifery",
    "public-health",

    # --- Faculty of Humanities, Arts and Social Sciences (HASS) ---
    "english", "english-film",
    "history",
    "philosophy",
    "classics", "classics-ancient-history",
    "modern-languages", "modernlanguages", "languages",
    "film", "film-studies",
    "drama",
    "communications",
    "archaeology",
    "theology-religion", "theology",
    "spa", "sociology-philosophy-anthropology",
    "sociology", "anthropology",
    "politics",
    "law",
    "education",
    "liberal-arts",
    "art-history-visual-culture", "art-history",
    "arab-islamic-studies", "iais",
    "media",
    "mining-engineering", "mining",

    # --- Penryn / Cornwall-specific programmes ---
    "ecology-conservation", "conservation-biology",
    "marine-biology", "zoology",
    "environmental-humanities",
    "human-geography", "physical-geography",
]
print(f"Candidates to probe: {len(CANDIDATE_SLUGS)}")

Candidates to probe: 74


## 3. Years to scrape

Six academic years (2019/20 through 2024/25), which straddles COVID. Exeter encodes these as `YYYY/Y` where `Y` is the last digit of the second year.

In [7]:
YEARS = ["2019/0", "2020/1", "2021/2", "2022/3", "2023/4", "2024/5"]
DISCOVERY_YEAR = "2024/5"
print(f"Will scrape {len(YEARS)} academic years.")

Will scrape 6 academic years.


## 4. Helpers: fetch, detect 'invalid department', parse module tables

In [9]:
def fetch_index_page(dept_slug: str, year: str) -> str | None:
    """Fetch a department index page. Returns HTML string, or None on failure."""
    params = {"prog": dept_slug, "year": year}
    try:
        r = requests.get(BASE_URL, params=params, headers=HEADERS, timeout=20)
    except requests.RequestException as e:
        print(f"  ! network error for {dept_slug} {year}: {e}")
        return None
    if r.status_code != 200:
        print(f"  ! HTTP {r.status_code} for {dept_slug} {year}")
        return None
    return r.text


INVALID_DEPT_MARKER = "invalid department has been provided"


def parse_index_html(html: str, dept_slug: str, year: str) -> pd.DataFrame:
    """Parse a department index page into a DataFrame of one row per module."""
    if html is None:
        return pd.DataFrame()
    if INVALID_DEPT_MARKER.lower() in html.lower():
        return pd.DataFrame()

    soup = BeautifulSoup(html, "html.parser")
    main = soup.find(id="main-col") or soup.find("main") or soup

    rows = []
    current_stage = None

    for element in main.find_all(["h2", "h3", "h4", "table"]):
        if element.name in ("h2", "h3", "h4"):
            text = element.get_text(" ", strip=True)
            if "module" in text.lower() and (
                "stage" in text.lower() or "master" in text.lower() or "postgraduate" in text.lower()
            ):
                current_stage = text
        elif element.name == "table":
            header_cells = [c.get_text(" ", strip=True).lower() for c in element.find_all("th")]
            if not ("code" in header_cells and any("title" in h for h in header_cells)):
                continue
            for tr in element.find_all("tr")[1:]:
                cells = tr.find_all(["td", "th"])
                if len(cells) < 4:
                    continue
                code = cells[0].get_text(" ", strip=True)
                title = cells[1].get_text(" ", strip=True)
                credits = cells[2].get_text(" ", strip=True)
                term = cells[3].get_text(" ", strip=True)
                if not re.match(r"^[A-Z]{3}[0-9MP][A-Z0-9]+", code):
                    continue
                rows.append({
                    "module_code": code,
                    "module_title": title,
                    "credits_raw": credits,
                    "term_raw": term,
                    "stage": current_stage or "",
                    "dept_slug": dept_slug,
                    "academic_year": year,
                })
    return pd.DataFrame(rows)

## 5. Discovery: probe every candidate and catalogue what it returns

For each candidate slug we fetch the 2024/5 index and record:
- `n_modules`: how many modules that slug listed
- `prefixes`: the **set of unique three-letter module-code prefixes** that appeared

The prefix set is what matters. If `management` returns `{BEE, BEM, BEA, BEF, BEP}` and `economics` returns `{BEE}`, both are kept but we'll prefer `management` where they overlap. If `communications` returns `{DRA}` and `drama` also returns `{DRA}`, we keep only one.

In [11]:
print(f"Discovering valid slugs by probing each candidate against {DISCOVERY_YEAR} ...\n")

slug_info = {}   # slug -> dict(n_modules, prefixes, df)

for slug in CANDIDATE_SLUGS:
    html = fetch_index_page(slug, DISCOVERY_YEAR)
    df = parse_index_html(html, slug, DISCOVERY_YEAR)
    if len(df):
        prefixes = sorted(df["module_code"].str[:3].unique().tolist())
        slug_info[slug] = {"n_modules": len(df), "prefixes": prefixes, "df": df}
        print(f"  ✓ {slug:35s}  {len(df):4d} modules  prefixes={prefixes}")
    else:
        print(f"  ✗ {slug}")
    time.sleep(SLEEP_BETWEEN_REQUESTS)

print(f"\n{len(slug_info)} valid slugs found.")

Discovering valid slugs by probing each candidate against 2024/5 ...

  ✓ economics                              96 modules  prefixes=['BEE']
  ✗ business
  ✗ business-school
  ✗ accounting
  ✗ finance
  ✓ management                            301 modules  prefixes=['BEM', 'BEP', 'BUS', 'MBA']
  ✗ marketing
  ✓ mathematics                            91 modules  prefixes=['EMP', 'MTH']
  ✗ maths
  ✗ computer-science
  ✗ computerscience
  ✗ cs
  ✓ engineering                           134 modules  prefixes=['ECM', 'EMP', 'ENG']
  ✓ physics                                52 modules  prefixes=['PHY']
  ✗ physics-astronomy
  ✓ geography                              78 modules  prefixes=['GEO']
  ✓ biosciences                            98 modules  prefixes=['BIO', 'JBI']
  ✗ biology
  ✓ naturalsciences                        19 modules  prefixes=['NSC']
  ✗ natural-sciences
  ✗ renewable-energy
  ✗ environmental-sciences
  ✗ med-school
  ✗ medicine
  ✗ medical-sciences
  ✗ medicalsciences
 

### 5a. Coverage-greedy de-duplication

We want the **smallest set of slugs** that covers **every distinct module-code prefix** we saw. A simple greedy pass does it: repeatedly pick the slug that adds the most new prefixes until all prefixes are covered.

In [13]:
all_prefixes = set()
for info in slug_info.values():
    all_prefixes.update(info["prefixes"])

print(f"Total distinct module-code prefixes seen: {len(all_prefixes)}")
print(f"Prefixes: {sorted(all_prefixes)}\n")

remaining = set(all_prefixes)
chosen_slugs = []

# Sort by number of prefixes (desc), then by module count (desc) — bigger slugs first.
slug_order = sorted(
    slug_info.items(),
    key=lambda kv: (-len(kv[1]['prefixes']), -kv[1]['n_modules']),
)

for slug, info in slug_order:
    new = set(info['prefixes']) & remaining
    if new:
        chosen_slugs.append(slug)
        remaining -= new
        print(f"  pick {slug:30s}  adds {sorted(new)}")
    if not remaining:
        break

print(f"\nChose {len(chosen_slugs)} slugs covering all {len(all_prefixes)} prefixes.")
print(f"Chosen slugs: {chosen_slugs}")

Total distinct module-code prefixes seen: 34
Prefixes: ['ANT', 'ARC', 'BEE', 'BEM', 'BEP', 'BIO', 'BUS', 'CLA', 'CMM', 'DRA', 'EAF', 'EAS', 'ECM', 'EDU', 'EFP', 'EMP', 'ENG', 'ERP', 'GEO', 'HIH', 'HIS', 'JBI', 'LAW', 'LIB', 'MBA', 'MTH', 'NSC', 'PHL', 'PHY', 'POL', 'PSY', 'PYC', 'SOC', 'THE']

  pick management                      adds ['BEM', 'BEP', 'BUS', 'MBA']
  pick engineering                     adds ['ECM', 'EMP', 'ENG']
  pick education                       adds ['EDU', 'EFP', 'ERP']
  pick communications                  adds ['CMM', 'DRA', 'EAF']
  pick psychology                      adds ['PSY', 'PYC']
  pick classics                        adds ['CLA', 'THE']
  pick biosciences                     adds ['BIO', 'JBI']
  pick mathematics                     adds ['MTH']
  pick history                         adds ['HIH', 'HIS']
  pick politics                        adds ['POL']
  pick law                             adds ['LAW']
  pick economics                       add

## 6. Full scrape across all years

In [15]:
DEPARTMENTS = {slug: slug.replace('-', ' ').title() for slug in chosen_slugs}
all_frames = []

for year in YEARS:
    year_frames = []
    print(f"\n=== Academic year {year} ===")
    for slug, display_name in DEPARTMENTS.items():
        html = fetch_index_page(slug, year)
        df = parse_index_html(html, slug, year)
        if len(df):
            print(f"  {slug:30s}  {len(df):4d} modules")
            df['dept_name'] = display_name
            year_frames.append(df)
        else:
            print(f"  {slug:30s}  (no data this year)")
        time.sleep(SLEEP_BETWEEN_REQUESTS)

    if year_frames:
        year_df = pd.concat(year_frames, ignore_index=True)
        out_path = RAW_DIR / f"index_{year.replace('/', '-')}.csv"
        year_df.to_csv(out_path, index=False)
        print(f"  -> wrote {len(year_df)} rows to {out_path.name}")
        all_frames.append(year_df)


=== Academic year 2019/0 ===
  management                        18 modules
  engineering                       88 modules
  education                         48 modules
  communications                  (no data this year)
  psychology                       124 modules
  classics                          94 modules
  biosciences                       71 modules
  mathematics                       60 modules
  history                           66 modules
  politics                          74 modules
  law                               45 modules
  economics                       (no data this year)
  english                           61 modules
  geography                         51 modules
  philosophy                        50 modules
  physics                         (no data this year)
  archaeology                       38 modules
  anthropology                      30 modules
  sociology                         38 modules
  naturalsciences                   17 modules
  liberal

## 7. Combine, de-duplicate, summarise

In [ ]:
if all_frames:
    index_all = pd.concat(all_frames, ignore_index=True)

    # Critical: a single module_code in one year can appear under several slugs (because
    # slugs overlap — e.g. Finance modules might show up under both 'management' and
    # some Finance-specific page). Drop dupes on (code, year), keeping whichever slug
    # came first in chosen_slugs — which by greedy order is the more-inclusive one.
    before = len(index_all)
    index_all = index_all.drop_duplicates(subset=['module_code', 'academic_year']).reset_index(drop=True)
    print(f"Dropped {before - len(index_all):,} duplicate (code, year) rows across slugs.")

    # Tag every module with its three-letter code prefix — this is the real department key.
    index_all['dept_code'] = index_all['module_code'].str[:3]

    out_path = RAW_DIR / 'index_all.csv'
    index_all.to_csv(out_path, index=False)
    print(f"Wrote master index: {out_path}  ({len(index_all):,} rows)")

    print('\nRows per year:')
    print(index_all['academic_year'].value_counts().sort_index())

    print('\nModule count by three-letter dept_code across all years:')
    print(index_all['dept_code'].value_counts())

    print('\n-- Distinct dept_codes seen per year (checks that coverage is stable) --')
    print(index_all.groupby('academic_year')['dept_code'].nunique())
else:
    print('No data collected — check that the discovery step found any valid slugs.')

## Next step

Proceed to **`02_scrape_module_details.ipynb`**, which reads `index_all.csv` and visits each individual module page to extract the assessment breakdown, teaching hours, cohort size, and number of assessment items. From notebook 02 onwards, `dept_code` (the three-letter prefix like `BEE`, `BEF`, `SPA`) is the real department identifier — `dept_slug` was only useful for navigating the module bank URL.
